In [ ]:
import copy
from functools import reduce

import matplotlib.pyplot as plt
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# 0. Device and seed
# ============================================================

torch.set_default_dtype(torch.float64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(42)


# ============================================================
# 1. Problem setting
# ============================================================

n = 4
m = 2
output_dim = n + m

mu = 1.0
rho = 0.5
T = 10.0

# Use a slightly longer run for the final comparison, and a short
# optimizer-calibration stage so QFM and FC do not have to share one LR.
epochs = 200
tune_epochs = 30
batch_size = 10

lr_candidates = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
optimizer_cls = optim.AdamW
optimizer_kwargs = {"weight_decay": 0.0}

n_qubits = output_dim
L = 2
entangling_layers_per_block = 2

fc_hidden_dim = 16
fc_depth = 1


def generate_full_row_rank_A(m, n):
    assert m <= n, "Require m <= n."

    while True:
        A_cpu = torch.randn(m, n, dtype=torch.float64)
        if torch.linalg.matrix_rank(A_cpu).item() == m:
            return A_cpu.to(device)


A = generate_full_row_rank_A(m, n)
A = A / torch.linalg.norm(A, dim=1, keepdim=True)

c = torch.randn(n, 1, device=device)

x_feasible = torch.randn(n, 1, device=device)
b = A @ x_feasible

x0 = torch.zeros(n, 1, device=device)
v0 = torch.zeros(m, 1, device=device)
y0 = torch.cat([x0, v0], dim=0).T

print("A shape:", A.shape)
print("b shape:", b.shape)
print("c shape:", c.shape)
print("y0 shape:", y0.shape)


# ============================================================
# 2. Objective and gradient
# ============================================================

def objective_f(x):
    return 0.5 * mu * torch.sum((x - c.T) ** 2, dim=1)


def grad_f_manual(x):
    return mu * (x - c.T)


# ============================================================
# 3. Closed-form KKT reference
# ============================================================

with torch.no_grad():
    AAT = A @ A.T
    rhs = mu * (A @ c - b)

    v_star = torch.linalg.solve(AAT, rhs)
    x_star = c - (1.0 / mu) * A.T @ v_star

    ref_obj = objective_f(x_star.T).item()

print("\nReference objective f(x_star):", ref_obj)


# ============================================================
# 4. Local observable components
# ============================================================

def Observable_local_components(n_qubits, device="cpu", dtype=torch.float64):
    P0 = torch.tensor(
        [[1.0, 0.0],
         [0.0, 0.0]],
        device=device,
        dtype=dtype,
    )
    I2 = torch.eye(2, device=device, dtype=dtype)

    observables = []

    for j in range(n_qubits):
        factors = [P0 if k == j else I2 for k in range(n_qubits)]
        Oj = reduce(lambda X, Y: torch.kron(X, Y), factors)
        observables.append(Oj)

    return observables


O_components_torch = Observable_local_components(
    n_qubits,
    device="cpu",
    dtype=torch.float64,
)

O_components = [Oj.numpy() for Oj in O_components_torch]


# ============================================================
# 5. Quantum Fourier model circuit
# ============================================================

dev = qml.device("default.qubit", wires=n_qubits)


def novel_exp_encoding_multi(t_scalar, layer):
    tau = t_scalar / T

    for j in range(n_qubits):
        phi = (j + 1) * torch.pi * tau
        coef = 3 ** layer

        qml.RZ(coef * phi, wires=j)
        qml.RY(coef * phi, wires=j)


@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit_QFM(t_scalar, weights):
    qml.templates.StronglyEntanglingLayers(
        weights[0],
        wires=range(n_qubits),
    )

    for layer in range(L):
        novel_exp_encoding_multi(t_scalar, layer)

        qml.templates.StronglyEntanglingLayers(
            weights[layer + 1],
            wires=range(n_qubits),
        )

    return [
        qml.expval(qml.Hermitian(O_components[j], wires=range(n_qubits)))
        for j in range(n_qubits)
    ]


# ============================================================
# 6. QFM-QPINN and fully-connected PINN models
# ============================================================

class QFM_QPINN(nn.Module):
    def __init__(self):
        super().__init__()

        self.weights = nn.Parameter(
            0.2 * torch.randn(
                L + 1,
                entangling_layers_per_block,
                n_qubits,
                3,
                dtype=torch.float64,
                device=device,
            )
        )

        self.readout = nn.Linear(
            n_qubits,
            output_dim,
            bias=True,
            dtype=torch.float64,
        )

        nn.init.xavier_uniform_(self.readout.weight)
        nn.init.zeros_(self.readout.bias)

        self.alpha = nn.Parameter(
            torch.tensor(0.5, dtype=torch.float64, device=device)
        )

    def quantum_forward_single(self, t_scalar):
        q_out = circuit_QFM(t_scalar, self.weights)
        q_out = torch.stack(q_out)

        q_out = 2.0 * q_out - 1.0

        return q_out

    def forward(self, t):
        outputs = []

        for i in range(t.shape[0]):
            t_scalar = t[i, 0]

            q_out = self.quantum_forward_single(t_scalar)
            raw = self.readout(q_out)

            alpha_pos = torch.nn.functional.softplus(self.alpha)
            envelope = 1.0 - torch.exp(-alpha_pos * t_scalar)

            y = y0.squeeze(0) + envelope * raw

            outputs.append(y)

        return torch.stack(outputs, dim=0)


class FC_PINN(nn.Module):
    def __init__(self, hidden_dim=16, depth=1):
        super().__init__()

        layers = []
        in_dim = 1

        for _ in range(depth):
            layers.append(nn.Linear(in_dim, hidden_dim, dtype=torch.float64))
            layers.append(nn.Tanh())
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, output_dim, dtype=torch.float64))

        self.net = nn.Sequential(*layers)

        self.alpha = nn.Parameter(
            torch.tensor(0.5, dtype=torch.float64, device=device)
        )

    def forward(self, t):
        tau = t / T
        raw = self.net(tau)

        alpha_pos = torch.nn.functional.softplus(self.alpha)
        envelope = 1.0 - torch.exp(-alpha_pos * t)

        y = y0 + envelope * raw

        return y


# ============================================================
# 7. Time derivative and QPINN loss
# ============================================================

def time_derivative(y, t):
    dy_dt = torch.zeros_like(y)

    for i in range(y.shape[1]):
        dy_dt[:, i:i + 1] = torch.autograd.grad(
            y[:, i:i + 1],
            t,
            grad_outputs=torch.ones_like(y[:, i:i + 1]),
            create_graph=True,
            retain_graph=True,
        )[0]

    return dy_dt


def qpinn_loss(model, t):
    y = model(t)

    x = y[:, :n]
    v = y[:, n:]

    dy_dt = time_derivative(y, t)

    dx_dt = dy_dt[:, :n]
    dv_dt = dy_dt[:, n:]

    grad_f = grad_f_manual(x)

    ATv = v @ A
    Ax_minus_b = x @ A.T - b.T

    r_x = dx_dt + rho * (grad_f + ATv)
    r_v = dv_dt - rho * Ax_minus_b

    loss_x = torch.mean(r_x ** 2)
    loss_v = torch.mean(r_v ** 2)

    loss = loss_x + loss_v

    return loss, loss_x, loss_v


# ============================================================
# 8. Evaluation helpers
# ============================================================

def evaluate_terminal_solution(model):
    with torch.no_grad():
        t_T = torch.tensor([[T]], dtype=torch.float64, device=device)
        y_T = model(t_T)

        x_T = y_T[:, :n].T
        v_T = y_T[:, n:].T

        x_error = torch.norm(x_T - x_star).item()
        objective = objective_f(x_T.T).item()
        objective_gap = objective - ref_obj
        feasibility_norm = torch.norm(A @ x_T - b).item()

    return {
        "x_T": x_T,
        "v_T": v_T,
        "x_error": x_error,
        "objective": objective,
        "objective_gap": objective_gap,
        "feasibility_norm": feasibility_norm,
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def safe_for_log(v, eps=1e-16):
    v = float(v)
    if v <= 0.0:
        return eps
    return v


def make_time_batches(num_epochs, batch_size, seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    batches = []
    for _ in range(num_epochs):
        t = torch.rand(
            batch_size,
            1,
            dtype=torch.float64,
            generator=generator,
        ).to(device) * T
        batches.append(t)

    return batches


train_batches = make_time_batches(epochs, batch_size, seed=2026)
tune_batches = make_time_batches(tune_epochs, batch_size, seed=2027)
validation_t = torch.linspace(
    0.0,
    T,
    25,
    dtype=torch.float64,
    device=device,
).reshape(-1, 1)


def residual_on_grid(model, t_grid):
    t = t_grid.detach().clone().requires_grad_(True)
    loss, loss_x, loss_v = qpinn_loss(model, t)
    return loss.item(), loss_x.item(), loss_v.item()


# ============================================================
# 9. Optimizer calibration and training
# ============================================================

def make_optimizer(model, lr):
    return optimizer_cls(model.parameters(), lr=lr, **optimizer_kwargs)


def short_train_score(base_model, lr, batches):
    model = copy.deepcopy(base_model).to(device)
    optimizer = make_optimizer(model, lr)

    for t_base in batches:
        optimizer.zero_grad()
        t = t_base.detach().clone().requires_grad_(True)
        loss, _, _ = qpinn_loss(model, t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

    val_loss, _, _ = residual_on_grid(model, validation_t)

    return val_loss


def calibrate_lr(base_model, name):
    print(f"\nCalibrating optimizer for {name} with {optimizer_cls.__name__}...")

    scores = []
    for lr in lr_candidates:
        val_loss = short_train_score(base_model, lr, tune_batches)
        scores.append((lr, val_loss))
        print(f"  lr={lr:.1e}, validation residual={val_loss:.6e}")

    best_lr, best_score = min(scores, key=lambda item: item[1])
    print(f"  selected lr={best_lr:.1e} for {name} (validation residual={best_score:.6e})")

    return best_lr, scores


def train_model(model, lr, name, batches):
    optimizer = make_optimizer(model, lr)

    history = {
        "loss": [],
        "loss_x": [],
        "loss_v": [],
        "x_error": [],
        "objective_gap": [],
        "feasibility_norm": [],
    }

    print(f"\nStart training {name} with lr={lr:.1e}...")

    for epoch, t_base in enumerate(batches, start=1):
        optimizer.zero_grad()

        t = t_base.detach().clone().requires_grad_(True)
        loss, loss_x, loss_v = qpinn_loss(model, t)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        info = evaluate_terminal_solution(model)

        history["loss"].append(safe_for_log(loss.item()))
        history["loss_x"].append(safe_for_log(loss_x.item()))
        history["loss_v"].append(safe_for_log(loss_v.item()))
        history["x_error"].append(safe_for_log(info["x_error"]))
        history["objective_gap"].append(abs(info["objective_gap"]) + 1e-16)
        history["feasibility_norm"].append(safe_for_log(info["feasibility_norm"]))

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"{name} | Epoch {epoch:03d}/{len(batches)}, "
                f"loss={loss.item():.6e}, "
                f"loss_x={loss_x.item():.6e}, "
                f"loss_v={loss_v.item():.6e}, "
                f"||x(T)-x*||={info['x_error']:.6e}"
            )

    print(f"Finished training {name}.")

    return history


# Use separate seeds for the two architectures, then keep each base copy fixed
# across its own LR candidates.
torch.manual_seed(123)
qfm_base = QFM_QPINN().to(device)

torch.manual_seed(456)
fc_base = FC_PINN(hidden_dim=fc_hidden_dim, depth=fc_depth).to(device)

qfm_params = count_parameters(qfm_base)
fc_params = count_parameters(fc_base)

print("\n================ Model Sizes ================")
print(f"QFM-QPINN trainable parameters: {qfm_params}")
print(f"FC-PINN trainable parameters:  {fc_params}")
print(f"Parameter ratio FC / QFM:      {fc_params / qfm_params:.2f}")

best_lr_qfm, qfm_lr_scores = calibrate_lr(qfm_base, "QFM-QPINN")
best_lr_fc, fc_lr_scores = calibrate_lr(fc_base, "FC-PINN")

qfm_model = copy.deepcopy(qfm_base).to(device)
fc_model = copy.deepcopy(fc_base).to(device)

histories = {
    "QFM-QPINN": train_model(qfm_model, best_lr_qfm, "QFM-QPINN", train_batches),
    "FC-PINN": train_model(fc_model, best_lr_fc, "FC-PINN", train_batches),
}

models = {
    "QFM-QPINN": qfm_model,
    "FC-PINN": fc_model,
}

final_infos = {name: evaluate_terminal_solution(model) for name, model in models.items()}

print("\n================ Final Comparison ================")
print("\nReference x_star:")
print(x_star.T.cpu())
print("Reference objective:", ref_obj)

for name, info in final_infos.items():
    print(f"\n{name} approximate x(T):")
    print(info["x_T"].T.cpu())
    print(f"{name} objective gap       = {info['objective_gap']:.12e}")
    print(f"{name} feasibility norm    = {info['feasibility_norm']:.12e}")
    print(f"{name} ||x(T) - x_star||   = {info['x_error']:.12e}")


# ============================================================
# 10. Publication-style plots
# ============================================================

def moving_average(values, window=9):
    values = torch.tensor(values, dtype=torch.float64)
    if values.numel() < window:
        return values.numpy()

    pad_left = window // 2
    pad_right = window - 1 - pad_left
    padded = torch.nn.functional.pad(values, (pad_left, pad_right), mode="replicate")
    kernel = torch.ones(window, dtype=torch.float64) / window
    smooth = torch.nn.functional.conv1d(
        padded.reshape(1, 1, -1),
        kernel.reshape(1, 1, -1),
    ).reshape(-1)

    return smooth.numpy()


plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
    "font.size": 11,
})

colors = {
    "QFM-QPINN": "#3561A7",
    "FC-PINN": "#C44E52",
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
epoch_axis = range(1, epochs + 1)

for name, hist in histories.items():
    color = colors[name]

    axes[0].semilogy(
        epoch_axis,
        hist["loss"],
        color=color,
        alpha=0.18,
        linewidth=1.0,
    )
    axes[0].semilogy(
        epoch_axis,
        moving_average(hist["loss"]),
        color=color,
        linewidth=2.4,
        label=f"{name} (lr={best_lr_qfm:.0e})" if name == "QFM-QPINN" else f"{name} (lr={best_lr_fc:.0e})",
    )

    axes[1].semilogy(
        epoch_axis,
        hist["x_error"],
        color=color,
        alpha=0.18,
        linewidth=1.0,
    )
    axes[1].semilogy(
        epoch_axis,
        moving_average(hist["x_error"]),
        color=color,
        linewidth=2.4,
        label=name,
    )

axes[0].set_title("ODE Residual Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel(r"$\mathcal{L}_{\mathrm{QPINN}}$")
axes[0].legend(loc="best")

axes[1].set_title("Terminal Solution Error")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel(r"$\|x_\theta(T)-x^\star\|_2$")
axes[1].legend(loc="best")

fig.suptitle("QFM-QPINN vs Fully Connected PINN with Calibrated AdamW", fontsize=13)
plt.show()


# ============================================================
# 11. Final solution comparison
# ============================================================

plt.figure(figsize=(8, 4.8), dpi=140)

component_axis = range(1, n + 1)

plt.plot(
    component_axis,
    x_star.detach().cpu().numpy().flatten(),
    "k--",
    linewidth=2.2,
    label=r"Reference $x^\star$",
)

markers = {
    "QFM-QPINN": "o-",
    "FC-PINN": "s-",
}

for name, info in final_infos.items():
    plt.plot(
        component_axis,
        info["x_T"].detach().cpu().numpy().flatten(),
        markers[name],
        color=colors[name],
        linewidth=2.0,
        markersize=5,
        label=rf"{name}, error={info['x_error']:.2e}",
    )

plt.title("Final Terminal Solution")
plt.xlabel("Component index")
plt.ylabel("Value")
plt.xticks(component_axis)
plt.legend(loc="best")
plt.tight_layout()
plt.show()
